# 09 – Finding 5: how many rejections do the flaws cause, and why didn't 98% accuracy catch it?
**Test:** for every applicant the model rejects, remove **one** flaw and score them again. If they are
now approved, that flaw alone caused the rejection.

In [1]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from common import load_data, predict

df = load_data()
df["label_rejected"] = df.decision == "reject"
_, df["model_rejected"] = predict(df)

In [2]:
flaws = {
    "zero existing loans (set to 1)": (df.existing_loans == 0, {"existing_loans": 1}),
    "income over Rs2.5L (set to 2.5L)": (df.monthly_income > 250000, {"monthly_income": 250000}),
    "pincode 41xxxx (moved to 560034)": (df.pincode // 10000 == 41, {"pincode": 560034}),
    "age under 25 (set to 25)": (df.age < 25, {"age": 25}),
}
df["caused_by_flaw"] = False
rows = []
for name, (has_flaw, fix) in flaws.items():
    hit = df[has_flaw & df.model_rejected]
    _, still_rejected = predict(hit.assign(**fix))
    df.loc[hit.index[~still_rejected], "caused_by_flaw"] = True
    rows.append({"flaw": name, "applicants": has_flaw.sum(), "rejected": len(hit),
                 "approved_after_fix": (~still_rejected).sum()})
table = pd.DataFrame(rows)
table

,flaw,applicants,rejected,approved_after_fix
0,zero existing loans (set to 1),108,108,63
1,income over Rs2.5L (set to 2.5L),23,23,5
2,pincode 41xxxx (moved to 560034),27,21,8
3,age under 25 (set to 25),67,53,16


In [3]:
n = df.caused_by_flaw.sum()
print(f"Model rejections: {df.model_rejected.sum()}. Caused by one flaw alone: {n} "
      f"({n / df.model_rejected.sum():.1%}).")
flawed = df[df.caused_by_flaw]
print(f"Model agrees with the labels on {(flawed.model_rejected == flawed.label_rejected).mean():.1%} "
      f"of those flaw-caused rejections.")

Model rejections: 355. Caused by one flaw alone: 91 (25.6%).
Model agrees with the labels on 98.9% of those flaw-caused rejections.


**Conclusion:** about 1 in 4 of the model's rejections is caused by one of the flaws alone, and the
model still "agrees with the labels" on nearly all of them. The validation labels contain the same
rules, so accuracy rewards the model for copying them. That's why 98% accuracy could not reveal any of
these problems.